# Notebook 02 — Índice vetorial: embeddings + FAISS

Segundo notebook do Tech Challenge 4. Transformamos cada review da base do
notebook 01 num **vetor de significado** e montamos um índice que, dada uma
pergunta, devolve as reviews **mais parecidas em sentido** — mesmo quando
não compartilham nenhuma palavra.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Setup: base e modelo | `preparacao.carregar_base`, `indice.carregar_modelo` | — |
| 1 | O que é um embedding | `indice.gerar_embeddings` | IA Gen 01 · PLN 04 |
| 2 | Embeddings da base inteira | `indice.construir_indice` | IA Gen 02 (slides 16–17) |
| 3 | O índice FAISS e a métrica | `indice.IndiceVetorial` | IA Gen 02 (slides 21–22) |
| 4 | Busca semântica | `indice.buscar_texto` | IA Gen 02 |
| 5 | Filtro por metadado | `buscar(..., permitidos=...)` | bônus do TC4 · IA Gen 05 |
| 6 | Limites da busca vetorial | — | IA Gen 02 (slide 23) |
| 7 | Gravar o índice e o manifesto | `indice.salvar_indice` | Deep Learning 08 (MLOps) |

**Modelo:** `paraphrase-multilingual-MiniLM-L12-v2` — o mesmo da IA Gen aula 02:
local, gratuito, multilíngue (entende português) e pequeno (384 dimensões).

> Pré-requisitos: ter rodado o notebook 01 (ou `python -m voc_rag.preparacao`) e
> `pip install -r requirements.txt`. Na primeira vez o modelo (~470 MB) é baixado
> do Hugging Face. Sem o notebook: `python -m voc_rag.indice`.

In [1]:
import inspect
import time

import numpy as np
import pandas as pd

from voc_rag import config, indice
from voc_rag.preparacao import carregar_base

pd.set_option("display.max_colwidth", 110)


def mostrar_codigo(funcao):
    """Imprime o código-fonte de uma função do src/, para lermos junto com o resultado."""
    print(inspect.getsource(funcao))


def mostrar(resultado, colunas=("nota", "categoria")):
    """Imprime um top-k: similaridade, review_id, metadados e texto."""
    for posicao, similaridade in resultado:
        linha = documentos.iloc[posicao]
        extras = " · ".join(f"{c}={linha[c]}" for c in colunas)
        print(f"  {similaridade:.3f}  [{linha['review_id'][:8]}] {extras}\n         {linha['texto'][:100]}")


documentos = carregar_base()
modelo = indice.carregar_modelo()
print(f"{len(documentos):,} documentos | modelo: {config.MODELO_EMBEDDINGS}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

42,138 documentos | modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## Seção 1 — O que é um embedding *(IA Gen aula 01 · PLN aula 04)*

Um embedding é uma lista de números (aqui, 384) que representa o **sentido** de
um texto. O modelo foi treinado para que textos com o mesmo sentido fiquem
**próximos** nesse espaço. Medimos proximidade com a **similaridade do cosseno**:
1 = mesma direção, 0 = nada a ver.

`gerar_embeddings` já devolve vetores **normalizados** (norma 1). Com isso o
cosseno vira um simples produto interno (`a @ b`) — é o truque que o FAISS usa.

**Observe** abaixo: as duas primeiras frases não têm quase nenhuma palavra em
comum e mesmo assim ficam próximas; a terceira fala de outra coisa.

In [2]:
mostrar_codigo(indice.gerar_embeddings)

frases = ["O produto não chegou até hoje",
          "Ainda estou esperando minha encomenda",
          "Adorei a qualidade do tecido"]
vetores_exemplo = indice.gerar_embeddings(modelo, frases)
print("formato:", vetores_exemplo.shape, "| norma de cada vetor:", np.linalg.norm(vetores_exemplo, axis=1).round(3))
print("primeiros 8 números da frase 1:", vetores_exemplo[0, :8].round(3))

similaridade = pd.DataFrame(vetores_exemplo @ vetores_exemplo.T, index=frases, columns=["f1", "f2", "f3"])
similaridade.round(2)

def gerar_embeddings(modelo, textos, tamanho_lote=TAMANHO_LOTE, mostrar_progresso=False):
    """Uma matriz (n_textos x dimensão), float32, com cada linha de norma 1.

    Normalizar faz o produto interno virar similaridade do cosseno (PLN aula 04).
    """
    vetores = modelo.encode(list(textos), batch_size=tamanho_lote, normalize_embeddings=True,
                            show_progress_bar=mostrar_progresso, convert_to_numpy=True)
    return np.asarray(vetores, dtype="float32")

formato: (3, 384) | norma de cada vetor: [1. 1. 1.]
primeiros 8 números da frase 1: [-0.048  0.007  0.02  -0.063  0.052 -0.019 -0.07   0.006]


,f1,f2,f3
O produto não chegou até hoje,1.00,0.36,0.05
Ainda estou esperando minha encomenda,0.36,1.00,0.03
Adorei a qualidade do tecido,0.05,0.03,1.00


## Seção 2 — Embeddings da base inteira *(IA Gen aula 02, slides 16–17)*

Cada `texto` da base (a versão com **limpeza leve** do notebook 01 — o modelo
prefere texto natural, com acentos e pontuação) vira um vetor. É a parte cara do
pipeline: com CPU, alguns minutos para ~42 mil reviews. O `encode` processa em
**lotes** (`config.TAMANHO_LOTE`) para equilibrar memória e velocidade.

Se o índice já foi gerado **para esta mesma versão da base**, reaproveitamos;
`carregar_indice` confere isso pelo hash do CSV gravado no manifesto.
Troque `RECALCULAR` para `True` para forçar.

In [3]:
RECALCULAR = False

try:
    if RECALCULAR:
        raise FileNotFoundError("recálculo pedido")
    indice_vetorial, manifesto = indice.carregar_indice()
    print("Índice reaproveitado do disco (mesma versão da base).")
except (FileNotFoundError, ValueError) as motivo:
    print(f"Gerando embeddings ({motivo})...")
    indice_vetorial, manifesto = indice.construir_indice(documentos, modelo, mostrar_progresso=True)

print(f"{manifesto['n_documentos']:,} vetores de {manifesto['dimensao']} dimensões "
      f"| embeddings em {manifesto['segundos_embedding']} s")
assert indice_vetorial.review_ids == documentos["review_id"].tolist(), "índice e base fora de ordem"

Índice reaproveitado do disco (mesma versão da base).
42,138 vetores de 384 dimensões | embeddings em 81.4 s


## Seção 3 — O índice FAISS e a métrica *(IA Gen aula 02, slides 21–22)*

Usamos `faiss.IndexFlatIP`: **Flat** = compara a pergunta com **todos** os
vetores (busca exata); **IP** = *inner product*, que com vetores normalizados é o
cosseno. Com ~42 mil vetores de 384 números isso leva poucos milissegundos.

Índices aproximados (HNSW, IVF) só compensam com milhões de vetores (slide 22);
aqui trocariam exatidão por uma velocidade de que não precisamos.

Sem o `faiss` instalado, a classe cai para numpy — mesmo resultado, como na aula 02.
O parâmetro `permitidos` é o filtro por metadado da Seção 5.

In [4]:
mostrar_codigo(indice.IndiceVetorial)
print("motor em uso:", indice_vetorial.motor)

class IndiceVetorial:
    """Busca exata por similaridade do cosseno, com FAISS ou, sem ele, com numpy."""

    def __init__(self, vetores, review_ids):
        if len(vetores) != len(review_ids):
            raise ValueError("cada vetor precisa de um review_id")
        self.vetores = np.ascontiguousarray(vetores, dtype="float32")
        self.review_ids = list(review_ids)
        try:
            import faiss

            self._faiss = faiss
            self.faiss = faiss.IndexFlatIP(self.vetores.shape[1])  # produto interno = cosseno
            self.faiss.add(self.vetores)
        except ImportError:  # mesmo resultado, só mais lento em bases grandes
            self.faiss = None

    @property
    def motor(self):
        return "faiss" if self.faiss is not None else "numpy"

    def buscar(self, vetor_consulta, k=10, permitidos=None):
        """Top-k como lista de (posição, similaridade), da mais parecida para a menos.

        `permitidos` (posições na base) restringe a busca A

## Seção 4 — Busca semântica *(IA Gen aula 02)*

Perguntas escritas como um analista escreveria, **sem copiar** as palavras dos
clientes. **Observe** se o top-5 traz reviews que dizem a mesma coisa com outras
palavras, e o tempo de cada busca.

In [5]:
PERGUNTAS = [
    "o pedido nunca foi entregue",
    "produto chegou com defeito ou danificado",
    "entrega mais rápida do que o esperado",
    "recebi um produto diferente do anunciado",
]
for pergunta in PERGUNTAS:
    inicio = time.perf_counter()
    resultado = indice.buscar_texto(indice_vetorial, modelo, pergunta, k=5)
    print(f'\n"{pergunta}"  ({(time.perf_counter() - inicio) * 1000:.0f} ms)')
    mostrar(resultado)


"o pedido nunca foi entregue"  (11 ms)
  0.845  [aed4d320] nota=1 · categoria=moveis_decoracao
         o pedido nao foi entregue
  0.819  [a76939c8] nota=1 · categoria=fashion_roupa_masculina
         O pedido não foi entregue
  0.759  [801b85d8] nota=1 · categoria=cama_mesa_banho
         Meus pedidos não foram entregues.
  0.746  [747fedbb] nota=5 · categoria=eletronicos
         Não foi entregue o meu pedido
  0.742  [f32fd5d3] nota=1 · categoria=informatica_acessorios
         Apesar de constar como entregue no histórico de pedidos, nada foi entregue.

"produto chegou com defeito ou danificado"  (11 ms)
  0.921  [cb770d3c] nota=2 · categoria=perfumaria
         produto veio danificado
  0.847  [556e5259] nota=1 · categoria=informatica_acessorios
         O produto veio estragado
  0.838  [ea0036b5] nota=1 · categoria=esporte_lazer
         produto veio com defeito
  0.824  [9da5990c] nota=2 · categoria=cool_stuff
         O produto foi entregue dentro do prazo porém um dos produt

## Seção 5 — Filtro por metadado *(bônus do TC4 · IA Gen aula 05)*

O bônus do TC4 pede filtros por nota, período, categoria e entrega. Há dois jeitos:

- **Pós-filtro:** buscar o top-k e depois descartar o que não passa no filtro. Se o
  filtro é restritivo, sobra pouco ou nada.
- **Pré-filtro:** dizer ao índice quais documentos são candidatos **antes** de
  ranquear. É o que `permitidos` faz (no FAISS, com um `IDSelectorBatch`).

O filtro é uma máscara do pandas sobre a base; `np.flatnonzero` transforma a
máscara nas posições permitidas.

In [6]:
pergunta = "o produto demorou demais para chegar"
filtro = (documentos["nota"] <= 2) & (documentos["categoria"] == "cama_mesa_banho")
permitidos = np.flatnonzero(filtro)
print(f'"{pergunta}" | filtro: nota <= 2 e categoria = cama_mesa_banho ({len(permitidos):,} candidatas)')

print("\nPRÉ-FILTRO (permitidos):")
mostrar(indice.buscar_texto(indice_vetorial, modelo, pergunta, k=5, permitidos=permitidos))

sem_filtro = indice.buscar_texto(indice_vetorial, modelo, pergunta, k=5)
sobrou = [(p, s) for p, s in sem_filtro if filtro.iloc[p]]
print(f"\nPÓS-FILTRO: do top-5 sem filtro, só {len(sobrou)} passa(m) no filtro.")

"o produto demorou demais para chegar" | filtro: nota <= 2 e categoria = cama_mesa_banho (1,139 candidatas)

PRÉ-FILTRO (permitidos):
  0.771  [d059af5c] nota=2 · categoria=cama_mesa_banho
         Muito demora para fazer a entrega do produto.
  0.743  [3b3281d0] nota=1 · categoria=cama_mesa_banho
         Demorou muito pra entregar.
  0.728  [378cd58e] nota=1 · categoria=cama_mesa_banho
         Demorou demais
  0.687  [20b5c041] nota=2 · categoria=cama_mesa_banho
         Achei que o prazo oferecido pela targaryen foi longo demais para uma mercadoria leve e pequena.
  0.635  [0995741b] nota=2 · categoria=cama_mesa_banho
         Demorou muito a entrega

PÓS-FILTRO: do top-5 sem filtro, só 0 passa(m) no filtro.


## Seção 6 — Limites da busca vetorial *(IA Gen aula 02, slide 23)*

Dois limites que motivam as próximas etapas:

1. **Sempre há um top-k.** Mesmo uma pergunta fora do escopo recebe 5 "respostas".
   Compare as similaridades: é o sinal que o RAG vai usar para **recusar** quando
   não há evidência (requisito 4 do TC4).
2. **Termos exatos.** Nomes próprios, siglas e palavras raras ("Correios", "nota
   fiscal") nem sempre pesam o suficiente no embedding. É aí que o **BM25** da
   Etapa 4 entra, e a **busca híbrida com RRF** junta os dois.

In [7]:
for pergunta in ["o pedido nunca foi entregue", "qual é a capital da França?"]:
    resultado = indice.buscar_texto(indice_vetorial, modelo, pergunta, k=5)
    print(f'"{pergunta}" -> similaridades do top-5: {[round(s, 2) for _, s in resultado]}')

print('\nTermo exato: "nota fiscal"')
mostrar(indice.buscar_texto(indice_vetorial, modelo, "nota fiscal", k=5))
contem = documentos["texto"].str.contains("nota fiscal", case=False)
print(f"\n{contem.sum():,} reviews contêm literalmente 'nota fiscal'.")

"o pedido nunca foi entregue" -> similaridades do top-5: [0.84, 0.82, 0.76, 0.75, 0.74]
"qual é a capital da França?" -> similaridades do top-5: [0.48, 0.43, 0.36, 0.35, 0.35]

Termo exato: "nota fiscal"


  0.880  [22c23de5] nota=2 · categoria=cool_stuff
         Preciso da nota fiscal
  0.759  [58dd0d28] nota=3 · categoria=informatica_acessorios
         faltou cupom fiscal
  0.741  [a0bd5636] nota=3 · categoria=moveis_escritorio
         Ficou faltando a nota fiscal
  0.741  [1d9be76d] nota=4 · categoria=beleza_saude
         NÃO VEIO A NOTA FISCAL
  0.737  [ecf4b5b8] nota=4 · categoria=informatica_acessorios
         Faltou a nota fiscal

378 reviews contêm literalmente 'nota fiscal'.


## Seção 7 — Gravar o índice e o manifesto *(Deep Learning aula 08: MLOps)*

Gravamos em `artifacts/indice_vetorial/`:

| Arquivo | Conteúdo |
|---|---|
| `embeddings.npy` | a matriz de vetores, na ordem da base |
| `review_ids.txt` | o `review_id` de cada linha da matriz: a ponte vetor → citação |
| `manifesto.json` | modelo, dimensão, hash da base, tempo e versões das bibliotecas |

O FAISS em si não é gravado: remontar um índice *flat* é instantâneo, e assim
não dependemos do formato binário do FAISS. O **manifesto** responde "de onde veio
este índice?" — se a base mudar, `carregar_indice` recusa o índice velho.

In [8]:
pasta = indice.salvar_indice(indice_vetorial.vetores, indice_vetorial.review_ids, manifesto)
print("Gravado em", pasta)
for arquivo in sorted(pasta.iterdir()):
    print(f"  {arquivo.name:16s} {arquivo.stat().st_size / 1e6:6.1f} MB")
print()
print(open(pasta / indice.ARQUIVO_MANIFESTO, encoding="utf-8").read())

Gravado em C:\Development\1IAST-Fase4\artifacts\indice_vetorial
  embeddings.npy     64.7 MB
  manifesto.json      0.0 MB
  review_ids.txt      1.4 MB

{
  "criado_em": "2026-10-27T19:59:01+00:00",
  "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "n_documentos": 42138,
  "dimensao": 384,
  "normalizado": true,
  "metrica": "cosseno (produto interno de vetores normalizados)",
  "base": "base_conhecimento_reviews.csv",
  "base_sha256": "e5787ef192771b4809e8688f76d3efd4e6aff1b94ba43c9548cbc1d7e4725b9c",
  "segundos_embedding": 81.4,
  "versoes": {
    "sentence-transformers": "6.2.0",
    "torch": "2.14.1",
    "faiss-cpu": "1.15.1",
    "numpy": "2.5.4"
  }
}


## Resumo das decisões

| Decisão | Por quê |
|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | o da IA Gen 02: local, gratuito, entende português, 384 dimensões |
| Embedding do `texto` (limpeza leve) | o modelo foi treinado com texto natural |
| Vetores normalizados + produto interno | vira similaridade do cosseno |
| `IndexFlatIP` (busca exata) | ~42 mil vetores: exato e rápido; HNSW/IVF só na escala de milhões |
| Pré-filtro por metadado (`permitidos`) | um filtro restritivo não esvazia o top-k |
| Gravar embeddings + manifesto, não o FAISS | o caro é o embedding; o manifesto garante que índice e base batem |

**Próximo (Etapa 4):** índice **BM25** sobre o `texto_lexical` e **busca híbrida
com RRF**, mais uma checagem de recuperação com perguntas de exemplo.